Actual notebook to generate all descriptors for all possible pairs of variables for each of the datasets involved - Train, Test, Dream3-10, Dream3-50, Netsim-5, Netsim-10. <br><br>
N.B. The true labels are present in these descriptor (`is_causal`) but when they will be passed to `D2CWrapper`, they will be replaced with the prediction from the fitted model. (`D2CWrapper` is the actual benchmark wrapper that would recompute all descriptors starting from observations only, but if we have computed them before, no need to repeat).

<br>
<b> !!! - This is computationally intense - !!! </b>

# Descriptors Computation
This notebook handles the descriptors computation. 
This procedure involves the following steps:
- For each variable, we estimate its Markov Blanket (MB) by selecting its lagged versions from one time step before and one after. 
- We standardize the time series to avoid varsortability
- Using the estimated MB, we compute a set of descriptors for all possible causal pairs (i.e., $ t-\tau \rightarrow t, \forall \tau$). that characterize the causal relationship between the variable pairs. These descriptors include conditional mutual information terms and other statistical properties that provide insights into the dependencies and interactions between the variables.
- For families of descriptors, we compute the quantiles of their empirical distributions. This step captures the distributional characteristics and aids in feature representation for the classifier.
- The computed descriptors and their quantiles are compiled into an input feature vector. This vector encapsulates the essential characteristics of the causal relationships and serves as the input for the classifier.
- For training data, each input vector is labeled as causal (1) or noncausal (0) based on the original selection criteria from the synthetic data's Directed Acyclic Graph (DAG). This labeling is crucial for supervised learning and model training.
- The labeled dataset, comprising the feature vectors, is used to train a classifier. The classifier learns to predict the likelihood of causal relationships based on the descriptors.
- For unseen time series data, the trained classifier predicts the probability of causal links for each pair of variables. The predictions are based on the computed descriptors for the test data.

The DataLoader class handles time series data and directed acyclic graphs (DAGs) to prepare for the following stage: the descriptors computation.
The preparation includes:
1. Creating lagged time series: This step involves generating lagged versions of the original time series data. Lagged time series help in capturing temporal dependencies and interactions between variables at different time steps.
2. Flattening the original dictionaries into coherent lists: The original data, which may be stored in nested dictionaries, is flattened into lists. This transformation ensures that the data is in a consistent and accessible format for further processing.
3. Renaming the nodes of the DAGs: The nodes of the Directed Acyclic Graphs (DAGs) are renamed to maintain consistency and clarity. This step is crucial for accurately representing the causal relationships between variables in the DAGs.


We are now ready for the core of our methodology: the D2C method. <br>
This method starts from a list of observations and dags and computes the corresponding descritpors, storing them in a dataframe. <br>
The D2C class gets the following arguments: 
- `observations` (list): List of observations (pd.DataFrame) corresponding to each DAG.
- `dags` (list): List of directed acyclic graphs (DAGs) representing causal relationships.
- `couples_to_consider_per_dag` (int, optional): To speedup, one can consider only a limited number of possible couples of variables. Couples are chosen to respect a specific ratio of causal/noncausal. Therefore, a DAG must be available: it can only be used for training data. For testing purposes only, we recommend using `D2CWrapper` instead. Defaults to -1 (all couples).  
- `n_variables` (int, optional): Number of variables in the time series. Defaults to 3.
- `maxlags` (int, optional): Maximum number of lags in the time series. Defaults to 3.
- `seed` (int, optional): Random seed for reproducibility. Defaults to 42.
- `n_jobs` (int, optional): Number of parallel jobs to run. Defaults to 1.



In [ ]:
from d2c.descriptors import D2C, DataLoader
N_VARS = 5
MAXLAGS = 3
N_JOBS = 50

dataloaders = {}
original_observations_training = {} 
lagged_flattened_observations_training = {} 
flattened_dags_training = {} 

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/training_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_training[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_training[error_dist] = dataloader.get_observations()
    flattened_dags_training[error_dist] = dataloader.get_dags()

original_observations_list_training = []
for obs_list in original_observations_training.values():
    original_observations_list_training.extend(obs_list) 

lagged_flattened_observations_list_training = []
for obs_list in lagged_flattened_observations_training.values():
    lagged_flattened_observations_list_training.extend(obs_list)

flattened_dags_list_training = []
for dags_list in flattened_dags_training.values():
    flattened_dags_list_training.extend(dags_list)

d2c_new = D2C(observations=lagged_flattened_observations_list_training,
    dags=flattened_dags_list_training, 
    couples_to_consider_per_dag=-1, 
    n_variables=N_VARS, 
    maxlags=MAXLAGS,
    seed=42,
    n_jobs=50,
    full=True,
    dynamic=True,
    mb_estimator='ts',
    )

d2c_new.initialize()

d2c_new.get_descriptors_df().to_pickle(f'data/descriptors/descriptors_df_train.pkl')

In [ ]:
dataloaders = {}
original_observations_testing = {} 
lagged_flattened_observations_testing = {} 
flattened_dags_testing = {}
true_causal_dfs = {}

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/testing_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_testing[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_testing[error_dist] = dataloader.get_observations()
    flattened_dags_testing[error_dist] = dataloader.get_dags()
    true_causal_dfs[error_dist] = dataloader.get_true_causal_dfs()

original_observations_list_testing = []
for obs_list in original_observations_testing.values():
    original_observations_list_testing.extend(obs_list) 

lagged_flattened_observations_list_testing = []
for obs_list in lagged_flattened_observations_testing.values():
    lagged_flattened_observations_list_testing.extend(obs_list)

flattened_dags_list_testing = []
for dags_list in flattened_dags_testing.values():
    flattened_dags_list_testing.extend(dags_list)

true_causal_dfs_list_testing = []
for causal_df in true_causal_dfs.values():
    true_causal_dfs_list_testing.extend(causal_df)

from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_list_testing,
        dags=flattened_dags_list_testing, 
        couples_to_consider_per_dag=-1, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle(f'data/descriptors/descriptors_df_test.pkl')

In [ ]:
from d2c.descriptors import DataLoader
N_JOBS = 50
MAXLAGS = 3
N_VARS = 5
dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/netsym/netsym_5.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_testing,
        dags=flattened_dags_testing, 
        couples_to_consider_per_dag=-1, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle(f'data/descriptors/descriptors_netsim_5.pkl')

In [ ]:
from d2c.descriptors import DataLoader
N_JOBS = 50
MAXLAGS = 3
N_VARS = 5
dataloader = DataLoader(n_variables = 10,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/netsym/netsym_10.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_testing,
        dags=flattened_dags_testing, 
        couples_to_consider_per_dag=-1, 
        n_variables=10, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle(f'data/descriptors/descriptors_netsim_10.pkl')

In [ ]:
from d2c.descriptors import DataLoader
N_JOBS = 50
MAXLAGS = 3
N_VARS = 5
dataloader = DataLoader(n_variables = 10,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/dream3/dream3_10.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_testing,
        dags=flattened_dags_testing, 
        couples_to_consider_per_dag=-1, 
        n_variables=10, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle(f'data/descriptors/descriptors_dream3_10.pkl')

In [ ]:
from d2c.descriptors import DataLoader
N_JOBS = 50
MAXLAGS = 3
N_VARS = 5
dataloader = DataLoader(n_variables = 50,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/dream3/dream3_50.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_testing,
        dags=flattened_dags_testing, 
        couples_to_consider_per_dag=-1, 
        n_variables=50, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle(f'data/descriptors/descriptors_dream3_50.pkl')